In [1]:
%%configure -f
{
  "conf": {
    "spark.pyspark.virtualenv.enabled": "true",
    "spark.pyspark.virtualenv.bin.path": "/usr/bin/virtualenv",
    "spark.pyspark.virtualenv.type": "native",
    "spark.pyspark.python": "/usr/bin/python3",
    "spark.executorEnv.PYSPARK_PYTHON": "/usr/bin/python3"
  }
}

ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
5,application_1780100949250_0006,pyspark,idle,Link,Link,None,


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("LogisticRegression").getOrCreate()

df = spark.read.csv("s3://csc555-emr-studio-lmedina/HW5/diabetes-data-set.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

VBox()

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
8,application_1780100949250_0009,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- chol: string (nullable = true)
 |-- stab.glu: integer (nullable = true)
 |-- hdl: string (nullable = true)
 |-- ratio: string (nullable = true)
 |-- glyhb: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- height: string (nullable = true)
 |-- weight: string (nullable = true)
 |-- frame: string (nullable = true)
 |-- bp.1s: string (nullable = true)
 |-- bp.1d: string (nullable = true)
 |-- bp.2s: string (nullable = true)
 |-- bp.2d: string (nullable = true)
 |-- waist: string (nullable = true)
 |-- hip: string (nullable = true)
 |-- time.ppn: string (nullable = true)
 |-- insurance: integer (nullable = true)
 |-- fh: integer (nullable = true)
 |-- smoking: integer (nullable = true)
 |-- diabetic: string (nullable = true)

+----+--------+---+-----------+-----------+---+------+------+------+------+-----+-----+-----+-----+-----+---+--------+---------+---+-------+--------+
|chol|stab.glu|hdl|      ratio|      glyhb|age|gender|

In [3]:
df2 = df.select("chol", "age", "gender", "height", "weight", "fh", "diabetic")
df2 = df2.dropna()
df2.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+------+------+------+---+--------+
|chol|age|gender|height|weight| fh|diabetic|
+----+---+------+------+------+---+--------+
| 203| 46|female|    62|   121|  0|      no|
| 165| 29|female|    64|   218|  0|      no|
| 228| 58|female|    61|   256|  0|      no|
|  78| 67|  male|    67|   119|  0|      no|
| 249| 64|  male|    68|   183|  0|     yes|
+----+---+------+------+------+---+--------+
only showing top 5 rows

In [4]:
from pyspark.sql.functions import when, col

df2 = df2.withColumn("diabetic", when(col("diabetic") == "yes", 1).otherwise(0))
df2.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+------+------+------+---+--------+
|chol|age|gender|height|weight| fh|diabetic|
+----+---+------+------+------+---+--------+
| 203| 46|female|    62|   121|  0|       0|
| 165| 29|female|    64|   218|  0|       0|
| 228| 58|female|    61|   256|  0|       0|
|  78| 67|  male|    67|   119|  0|       0|
| 249| 64|  male|    68|   183|  0|       1|
+----+---+------+------+------+---+--------+
only showing top 5 rows

In [5]:
df2 = df2.withColumn("gender", when(col("gender") == "female", 1).otherwise(0))
df2.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+------+------+------+---+--------+
|chol|age|gender|height|weight| fh|diabetic|
+----+---+------+------+------+---+--------+
| 203| 46|     1|    62|   121|  0|       0|
| 165| 29|     1|    64|   218|  0|       0|
| 228| 58|     1|    61|   256|  0|       0|
|  78| 67|     0|    67|   119|  0|       0|
| 249| 64|     0|    68|   183|  0|       1|
+----+---+------+------+------+---+--------+
only showing top 5 rows

In [6]:
df2 = df2.withColumn("heightsi", col("height") * 0.0254)
df2 = df2.withColumn("weightsi", col("weight") * 0.45359)
df2.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+------+------+------+---+--------+------------------+------------------+
|chol|age|gender|height|weight| fh|diabetic|          heightsi|          weightsi|
+----+---+------+------+------+---+--------+------------------+------------------+
| 203| 46|     1|    62|   121|  0|       0|            1.5748|54.884389999999996|
| 165| 29|     1|    64|   218|  0|       0|            1.6256|          98.88262|
| 228| 58|     1|    61|   256|  0|       0|1.5493999999999999|         116.11904|
|  78| 67|     0|    67|   119|  0|       0|            1.7018|          53.97721|
| 249| 64|     0|    68|   183|  0|       1|1.7271999999999998|          83.00697|
+----+---+------+------+------+---+--------+------------------+------------------+
only showing top 5 rows

In [7]:
df2 = df2.withColumn("BMI", col("weightsi") / (col("heightsi") ** 2))
df2.show(5)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----+---+------+------+------+---+--------+------------------+------------------+------------------+
|chol|age|gender|height|weight| fh|diabetic|          heightsi|          weightsi|               BMI|
+----+---+------+------+------+---+--------+------------------+------------------+------------------+
| 203| 46|     1|    62|   121|  0|       0|            1.5748|54.884389999999996|  22.1308466810482|
| 165| 29|     1|    64|   218|  0|       0|            1.6256|          98.88262| 37.41903504314821|
| 228| 58|     1|    61|   256|  0|       0|1.5493999999999999|         116.11904| 48.37002740385487|
|  78| 67|     0|    67|   119|  0|       0|            1.7018|          53.97721|18.637746230716342|
| 249| 64|     0|    68|   183|  0|       1|1.7271999999999998|          83.00697|27.824623880216624|
+----+---+------+------+------+---+--------+------------------+------------------+------------------+
only showing top 5 rows

In [8]:
train, test = df2.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train.count())
print("Test rows:", test.count())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Training rows: 345
Test rows: 58

In [9]:
from pyspark.sql.types import FloatType
df2 = df2.withColumn("chol", col("chol").cast(FloatType()))
train, test = df2.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train.count())
print("Test rows:", test.count())

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Training rows: 345
Test rows: 58

In [10]:
sc.install_pypi_package("numpy")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.3.3 requires tzdata>=2022.7, which is not installed.
matplotlib 3.9.4 requires contourpy>=1.0.1, which is not installed.
matplotlib 3.9.4 requires cycler>=0.10, which is not installed.
matplotlib 3.9.4 requires fonttools>=4.22.0, which is not installed.
matplotlib 3.9.4 requires importlib-resources>=3.2.0; python_version < "3.10", which is not installed.
matplotlib 3.9.4 requires kiwisolver>=1.3.1, which is not installed.
matplotlib 3.9.4 requires pillow>=8, which is not installed.
pandas 2.3.3 requires python-dateutil>=2.8.2, but you have python-dateutil 2.8.1 which is incompatible.

In [11]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
assembler = VectorAssembler(
    inputCols=["age", "gender", "fh", "BMI", "chol"],
    outputCol="features",
    handleInvalid="skip"
)
train_assembled = assembler.transform(train)
test_assembled = assembler.transform(test)
lr = LogisticRegression(featuresCol="features", labelCol="diabetic")
model = lr.fit(train_assembled)

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [12]:
predictions = model.transform(test_assembled)
predictions.select("features", "diabetic", "prediction").show(10)
correct = predictions.filter(col("diabetic") == col("prediction")).count()
total = predictions.count()
accuracy = correct / total * 100
print(f"Accuracy: {accuracy:.2f}%")

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+--------------------+--------+----------+
|            features|diabetic|prediction|
+--------------------+--------+----------+
|[56.0,0.0,0.0,19....|       0|       0.0|
|[28.0,1.0,0.0,34....|       0|       0.0|
|[22.0,1.0,0.0,25....|       0|       0.0|
|[68.0,0.0,0.0,24....|       0|       0.0|
|[19.0,1.0,0.0,26....|       0|       0.0|
|[26.0,1.0,1.0,31....|       0|       0.0|
|[32.0,1.0,0.0,25....|       0|       0.0|
|[68.0,1.0,0.0,37....|       0|       0.0|
|[76.0,0.0,0.0,30....|       0|       0.0|
|[30.0,1.0,0.0,25....|       0|       0.0|
+--------------------+--------+----------+
only showing top 10 rows

Accuracy: 83.93%

In [13]:
#done

VBox()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…